# Risk sharing — Day 2
Run **Runtime > Run all** in Colab. Read one numbered section at a time.

This short notebook explains the two-agent practice example. Detailed validation
is in `validate_day2.py` in the repository. This example stays **provisional** until
Parsa confirms Day 1 and reviews the Day 2 derivations and numerical evidence.

Saved outputs come from a local checked run. Rerun in your own Colab session.
If packages are missing, run `%pip install numpy scipy` in a separate cell.

## 1. Set up the tiny example
An **agent** receives a share. A **wealth state** is one possible total amount,
ordered from low to high. These are possible scenarios, not payments over time.

Rows represent three wealth states. Columns represent two agents, with utilities
h1 and h2. Equal sharing starts each agent at half the wealth. Pareto weights say
how strongly each agent's utility counts; probability weights say how strongly
each state counts. Rank and wealth are separate arrays even though equal here.

The explicit inputs below match the provisional Day 1 illustration. The full
validation script reads `config.json`; update both after agreed specification changes.

In [1]:
import numpy as np
from scipy.optimize import linprog

ranks = (np.arange(3) + 0.5) / 3
wealth = ranks.copy()
probabilities = np.full(3, 1 / 3)
pareto_weights = np.array([1.0, 1.0])
consumption = np.column_stack((wealth / 2, wealth / 2))
print("Wealth:", wealth)
print("Equal shares:\n", consumption)

Wealth: [0.16666667 0.5        0.83333333]
Equal shares:
 [[0.08333333 0.08333333]
 [0.25       0.25      ]
 [0.41666667 0.41666667]]


## 2. Calculate welfare
**Welfare** scores the combined benefit of an allocation. Utilities express the
benefit of each agent's consumption. We multiply by agent weights and state
probabilities, then add the contributions.

$$h_1(r,c)=\log(1+\sqrt r+c),\quad h_2(r,c)=\log(1+r^{2/3}+c).$$
$$F(c)=\sum_{j,i}p_j\lambda_i h_i(r_j,c_{ji}).$$

`probabilities[:, None]` makes a column so each state's probability multiplies
both agents' utility entries. `log1p(x)` computes the natural logarithm of `1+x`.

In [2]:
def objective(c):
    utilities = np.column_stack((
        np.log1p(np.sqrt(ranks) + c[:, 0]),
        np.log1p(ranks ** (2 / 3) + c[:, 1]),
    ))
    return float(np.sum(probabilities[:, None] * pareto_weights * utilities))

print("Equal-sharing welfare:", objective(consumption))

Equal-sharing welfare: 1.2362178075439176


## 3. Calculate the gradient
The **gradient** estimates the welfare gain from a tiny increase in one
consumption entry, with all other entries fixed. A feasible redistribution also
changes another agent's entry; a directional derivative accounts for both.

$$g_{ji}=\frac{p_j\lambda_i}{1+\sqrt{r_j}+c_{ji}}\quad(h_1),\qquad
g_{ji}=\frac{p_j\lambda_i}{1+r_j^{2/3}+c_{ji}}\quad(h_2).$$

The probability and Pareto factors are included in these gradients.

In [3]:
def gradient(c):
    denominators = np.column_stack((
        1 + np.sqrt(ranks) + c[:, 0],
        1 + ranks ** (2 / 3) + c[:, 1],
    ))
    return probabilities[:, None] * pareto_weights / denominators

g = gradient(consumption)
print("Gradient:\n", g)

Gradient:
 [[0.22347643 0.24046784]
 [0.17031944 0.17730869]
 [0.14308991 0.14478807]]


## 4. Add tail scores
An increment assigned at one wealth state carries through to all higher wealth
states. Its **tail score** adds the current gradient and every gradient below it
in the low-to-high table:
$$H_{ji}=\sum_{k=j}^{m-1}g_{ki}.$$

For invented rows `(4,3), (1,2), (2,1)`, the score rows are
`(7,6), (3,3), (2,1)`. At the middle row, the local gradients favor agent 2,
but the tail scores tie. The reversal lets `cumsum` start from the highest state.

In [4]:
def tail_scores(g):
    return np.cumsum(g[::-1], axis=0)[::-1]

scores = tail_scores(g)
print("Tail scores:\n", scores)

Tail scores:
 [[0.53688578 0.56256459]
 [0.31340935 0.32209676]
 [0.14308991 0.14478807]]


## 5. Let the linear oracle assign increments
The **oracle** is a helper that solves the current linear approximation exactly.
Give each wealth increment to the highest-tail-score agent, splitting equally
among exact ties. Add increments cumulatively to recover consumption.

The first increment starts at zero wealth. The oracle gives a candidate allocation;
the optimizer's step size and repeated updates belong to Day 3.

In [5]:
def linear_oracle(g, wealth):
    scores = tail_scores(g)
    wealth_increments = np.diff(np.r_[0.0, wealth])
    winners = scores == scores.max(axis=1, keepdims=True)
    increments = wealth_increments[:, None] * winners / winners.sum(axis=1, keepdims=True)
    return np.cumsum(increments, axis=0)

candidate = linear_oracle(g, wealth)
print("Oracle candidate:\n", candidate)

Oracle candidate:
 [[0.         0.16666667]
 [0.         0.5       ]
 [0.         0.83333333]]


## 6. Check feasibility
**Budget:** each row distributes exactly the available wealth.
**Nonnegativity:** nobody receives negative consumption.
**Monotonicity:** every agent's share stays the same or increases at higher wealth.

For a $100/$200/$300 bonus, Alice's shares $50/$80/$80 are monotone. Shares
$50/$40/$80 would violate the rule. The first increment below anchors consumption at zero.

In [6]:
def check_feasibility(c):
    increments = np.diff(np.vstack((np.zeros((1, 2)), c)), axis=0)
    np.testing.assert_allclose(c.sum(axis=1), wealth, rtol=0, atol=1e-10)
    if c.min() < -1e-12 or increments.min() < -1e-12:
        raise ValueError("Negative consumption or decreasing share")

check_feasibility(consumption)
check_feasibility(candidate)
print("PASS: both allocations satisfy all three rules.")

PASS: both allocations satisfy all three rules.


## 7. Check the gradient numerically
Move a tiny amount toward the oracle candidate and compare the objective change
with the gradient prediction. For this equal-sharing start, both positive and
negative small moves are feasible. This is one centered finite-difference check;
the separate validation suite covers more cases and boundaries.

In [7]:
direction = candidate - consumption
step = 1e-5
check_feasibility(consumption + step * direction)
check_feasibility(consumption - step * direction)
predicted = float(np.sum(g * direction))
measured = (objective(consumption + step * direction) - objective(consumption - step * direction)) / (2 * step)
gradient_error = abs(predicted - measured)
np.testing.assert_allclose(predicted, measured, rtol=1e-6, atol=1e-7)
print("Gradient check error:", gradient_error)

Gradient check error: 2.518804175649647e-12


## 8. Independently check the oracle
HiGHS solves the same linear problem using consumption variables directly.
The equalities enforce budgets; the inequalities enforce monotonicity; bounds
enforce nonnegativity. This reference formulation does not use tail scores.

Compare the optimal **linear objective values**, because tied optima can have
different allocations. Read this cell after the main concepts are clear.

In [8]:
m, n = consumption.shape
budgets = np.zeros((m, m * n))
for j in range(m):
    budgets[j, j * n:(j + 1) * n] = 1
monotonicity = np.zeros(((m - 1) * n, m * n))
for j in range(m - 1):
    for i in range(n):
        monotonicity[j * n + i, j * n + i] = 1
        monotonicity[j * n + i, (j + 1) * n + i] = -1
lp = linprog(-g.ravel(), A_eq=budgets, b_eq=wealth,
             A_ub=monotonicity, b_ub=np.zeros((m - 1) * n),
             bounds=(0, None), method="highs")
if not lp.success:
    raise RuntimeError(lp.message)
check_feasibility(lp.x.reshape(m, n))
oracle_error = abs(float(np.sum(g * candidate)) + lp.fun)
np.testing.assert_allclose(float(np.sum(g * candidate)), -lp.fun, rtol=1e-9, atol=1e-9)
print("Oracle/HiGHS value difference:", oracle_error)
print("Computational checks passed; joint review remains pending.")

Oracle/HiGHS value difference: 0.0
Computational checks passed; joint review remains pending.


## 9. Save results
This last cell saves the small example and versions into `risk_sharing_day2/`
and `day2_results.zip`. Download the zip in Colab's Files panel and save the notebook.
It does not upload to GitHub or change the human review record.

In [9]:
#@title Save the example and environment record
import json
import platform
import sys
import zipfile
from pathlib import Path
from importlib.metadata import version

out = Path("risk_sharing_day2")
out.mkdir(exist_ok=True)
summary = {"status": "PROVISIONAL: joint review pending",
           "equal_welfare": objective(consumption), "gradient_error": gradient_error,
           "oracle_lp_value_difference": oracle_error}
(out / "summary.json").write_text(json.dumps(summary, indent=2))
(out / "environment.json").write_text(json.dumps({"python": sys.version,
    "platform": platform.platform(), "packages": {name: version(name) for name in ("numpy", "scipy")}}, indent=2))
np.savetxt(out / "oracle.csv", np.column_stack((wealth, candidate)), delimiter=",",
           header="wealth,agent1,agent2", comments="")
with zipfile.ZipFile("day2_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ("summary.json", "environment.json", "oracle.csv"):
        archive.write(out / name, arcname="risk_sharing_day2/" + name)
print("Saved: risk_sharing_day2/ and day2_results.zip")

Saved: risk_sharing_day2/ and day2_results.zip


The joint Day 2 gate still needs Parsa's independent derivation and review of the
gradient, tail-score example, tolerances and LP comparisons. The full suite and
recorded evidence are in the repository; numerical checks do not establish a
nonlinear optimization certificate or continuous-distribution accuracy.

Sources: five-day plan p. 2; replication guide pp. 4-6; Carlier-Lachapelle (2011)
Example 2 p. 12 and adjoint formula (3.5) p. 6.